<a href="https://colab.research.google.com/github/jrangelg/Artificial-intelligence/blob/main/SURVEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SURVEY

In [10]:
options(survey.lonely.psu = "adjust")

if (!require("survey")) install.packages("survey", repos = "https://cloud.r-project.org")
library(survey)

# 1. Carga y preparación COMPLETA del dataset
data(mtcars)
datos <- mtcars
datos$id <- 1:nrow(datos)

set.seed(123)
datos$estrato <- sample(1:3, nrow(datos), replace = TRUE)
datos$conglomerado <- sample(1:8, nrow(datos), replace = TRUE)
datos$peso <- runif(nrow(datos), 1, 5)
datos$cluster1 <- sample(1:4, nrow(datos), replace = TRUE)
datos$cluster2 <- sample(1:8, nrow(datos), replace = TRUE)

# Crear variables derivadas AQUÍ (antes de definir los diseños)
datos$alto_mpg <- ifelse(datos$mpg > median(datos$mpg), 1, 0)

# 2. Diseños Muestrales (Ahora todos incluyen 'alto_mpg')

diseno_mas <- svydesign(ids = ~1, data = datos)
diseno_mas_pesos <- svydesign(ids = ~1, weights = ~peso, data = datos)
diseno_estratificado <- svydesign(ids = ~1, strata = ~estrato, weights = ~peso, data = datos)
diseno_conglomerados <- svydesign(ids = ~conglomerado, data = datos)
diseno_conglomerados_pesos <- svydesign(ids = ~conglomerado, weights = ~peso, data = datos)
diseno_estratificado_conglomerados <- svydesign(
  ids = ~conglomerado,
  strata = ~estrato,
  weights = ~peso,
  data = datos,
  nest = TRUE
)
diseno_multietapico <- svydesign(
  ids = ~cluster1 + cluster2,
  strata = ~estrato,
  weights = ~peso,
  data = datos,
  nest = TRUE
)

# 3. Estimaciones (Ya no darán error)

# Medias
print(svymean(~mpg, diseno_mas))
print(svymean(~mpg, diseno_mas_pesos))
print(svymean(~mpg, diseno_estratificado))
print(svymean(~mpg, diseno_conglomerados))
print(svymean(~mpg, diseno_conglomerados_pesos))
print(svymean(~mpg, diseno_estratificado_conglomerados))
print(svymean(~mpg, diseno_multietapico))

# Proporciones y Totales
print(svymean(~alto_mpg, diseno_estratificado))
print(svytotal(~mpg, diseno_estratificado))
print(svyby(~mpg, ~factor(cyl), diseno_estratificado, svymean))

# 4. Modelos Estadísticos
modelo <- svyglm(mpg ~ wt + hp + factor(cyl), design = diseno_estratificado)
print(summary(modelo))

modelo_logistico <- svyglm(
  alto_mpg ~ wt + hp,
  design = diseno_estratificado,
  family = quasibinomial()
)
print(summary(modelo_logistico))

Warning message in svydesign.default(ids = ~1, data = datos):
“No weights or probabilities supplied, assuming equal probability”
Warning message in svydesign.default(ids = ~conglomerado, data = datos):
“No weights or probabilities supplied, assuming equal probability”


      mean     SE
mpg 20.091 1.0654
     mean     SE
mpg 21.16 1.3429
     mean     SE
mpg 21.16 1.3729
      mean     SE
mpg 20.091 1.0732
     mean     SE
mpg 21.16 1.3581
     mean     SE
mpg 21.16 1.5002
     mean     SE
mpg 21.16 1.2219
            mean     SE
alto_mpg 0.54602 0.0971
    total     SE
mpg  1949 219.39
  factor(cyl)      mpg        se
4           4 27.60732 1.4819729
6           6 20.03350 0.4932298
8           8 15.04281 0.8190812

Call:
svyglm(formula = mpg ~ wt + hp + factor(cyl), design = diseno_estratificado)

Survey design:
svydesign(ids = ~1, strata = ~estrato, weights = ~peso, data = datos)

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept)  37.36595    2.51056  14.884 6.25e-14 ***
wt           -3.49853    0.72057  -4.855 5.42e-05 ***
hp           -0.02458    0.01077  -2.282   0.0313 *  
factor(cyl)6 -3.57895    1.38774  -2.579   0.0162 *  
factor(cyl)8 -3.02859    2.58957  -1.170   0.2532    
---
Signif. codes:  0 ‘***’ 0.001 ‘

Warning message:
“glm.fit: algorithm did not converge”



Call:
svyglm(formula = alto_mpg ~ wt + hp, design = diseno_estratificado, 
    family = quasibinomial())

Survey design:
svydesign(ids = ~1, strata = ~estrato, weights = ~peso, data = datos)

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)  6.247e+02  1.839e+01   33.96   <2e-16 ***
wt          -1.765e+02  5.279e+00  -33.44   <2e-16 ***
hp          -3.308e-01  9.991e-03  -33.11   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for quasibinomial family taken to be 7.724956e-11)

Number of Fisher Scoring iterations: 25



In [11]:
# 1. Ajustes globales, instalación y carga
options(survey.lonely.psu = "adjust")

if (!require("survey")) install.packages("survey", repos = "https://cloud.r-project.org")
library(survey)

# 2. Preparación de la estructura de datos muestral (Variables creadas al inicio)
datos <- iris
datos$id <- 1:nrow(datos)
datos$estrato <- datos$Species  # Estratos basados en la especie

set.seed(123)
datos$conglomerado <- sample(1:10, nrow(datos), replace = TRUE)
datos$cluster1 <- sample(1:5, nrow(datos), replace = TRUE)
datos$cluster2 <- sample(1:15, nrow(datos), replace = TRUE)
datos$peso <- runif(nrow(datos), 0.5, 3.5)
datos$sepalo_grande <- ifelse(datos$Sepal.Length > median(datos$Sepal.Length), 1, 0)

# 3. Diseños de Muestreo

# Muestreo aleatorio simple (MAS)
diseno_mas <- svydesign(ids = ~1, data = datos)
print(svymean(~Sepal.Length, diseno_mas))

# MAS con pesos
diseno_mas_pesos <- svydesign(ids = ~1, weights = ~peso, data = datos)
print(svymean(~Sepal.Length, diseno_mas_pesos))

# Muestreo estratificado
diseno_estratificado <- svydesign(ids = ~1, strata = ~estrato, weights = ~peso, data = datos)
print(svymean(~Sepal.Length, diseno_estratificado))

# Muestreo por conglomerados
diseno_conglomerados <- svydesign(ids = ~conglomerado, data = datos)
print(svymean(~Sepal.Length, diseno_conglomerados))

# Muestreo por conglomerados con pesos
diseno_conglomerados_pesos <- svydesign(ids = ~conglomerado, weights = ~peso, data = datos)
print(svymean(~Sepal.Length, diseno_conglomerados_pesos))

# Muestreo estratificado y por conglomerados
diseno_estratificado_conglomerados <- svydesign(
  ids = ~conglomerado,
  strata = ~estrato,
  weights = ~peso,
  data = datos,
  nest = TRUE
)
print(svymean(~Sepal.Length, diseno_estratificado_conglomerados))

# Muestreo multietápico
diseno_multietapico <- svydesign(
  ids = ~cluster1 + cluster2,
  strata = ~estrato,
  weights = ~peso,
  data = datos,
  nest = TRUE
)
print(svymean(~Sepal.Length, diseno_multietapico))

# 4. Estimaciones

# Estimación de proporciones
print(svymean(~sepalo_grande, diseno_estratificado))

# Estimación de totales
print(svytotal(~Sepal.Length, diseno_estratificado))

# Estimación de medias por dominio (por especie)
print(svyby(~Sepal.Length, ~Species, diseno_estratificado, svymean))

# 5. Modelado Estadístico

# Regresión lineal con diseño muestral
modelo_lineal <- svyglm(Sepal.Length ~ Sepal.Width + Petal.Length, design = diseno_estratificado)
print(summary(modelo_lineal))

# Regresión logística con diseño muestral
modelo_logistico <- svyglm(
  sepalo_grande ~ Sepal.Width + Petal.Length,
  design = diseno_estratificado,
  family = quasibinomial()
)
print(summary(modelo_logistico))

Warning message in svydesign.default(ids = ~1, data = datos):
“No weights or probabilities supplied, assuming equal probability”


               mean     SE
Sepal.Length 5.8433 0.0676
               mean    SE
Sepal.Length 5.8563 0.076
               mean     SE
Sepal.Length 5.8563 0.0532


Warning message in svydesign.default(ids = ~conglomerado, data = datos):
“No weights or probabilities supplied, assuming equal probability”


               mean     SE
Sepal.Length 5.8433 0.0811
               mean     SE
Sepal.Length 5.8563 0.0943
               mean     SE
Sepal.Length 5.8563 0.0812
               mean     SE
Sepal.Length 5.8563 0.0869
                 mean     SE
sepalo_grande 0.45623 0.0333
              total     SE
Sepal.Length 1747.5 65.911
              Species Sepal.Length         se
setosa         setosa     5.017622 0.04902150
versicolor versicolor     5.956662 0.08532259
virginica   virginica     6.630469 0.10424309

Call:
svyglm(formula = Sepal.Length ~ Sepal.Width + Petal.Length, design = diseno_estratificado)

Survey design:
svydesign(ids = ~1, strata = ~estrato, weights = ~peso, data = datos)

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept)   2.18077    0.25345   8.604 1.16e-14 ***
Sepal.Width   0.61589    0.07000   8.799 3.75e-15 ***
Petal.Length  0.47732    0.01791  26.657  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Disp

In [12]:
# 1. Ajustes globales, instalación y carga
options(survey.lonely.psu = "adjust")

if (!require("survey")) install.packages("survey", repos = "https://cloud.r-project.org")
if (!require("ggplot2")) install.packages("ggplot2", repos = "https://cloud.r-project.org")
library(survey)
library(ggplot2)

# Submuestra reproducible
set.seed(42)
datos <- diamonds[sample(1:nrow(diamonds), 1000), ]
datos$id <- 1:nrow(datos)

# Variables de diseño muestral
datos$estrato <- datos$clarity              # Estratificación
datos$psu <- as.numeric(datos$cut)          # UPM (Etapa 1)
datos$ssu <- as.numeric(datos$color)        # USM (Etapa 2)
datos$peso <- runif(nrow(datos), 0.8, 4.2)  # Ponderadores muestrales
datos$alta_gama <- ifelse(datos$price > median(datos$price), 1, 0) # Variable binaria

# 2. Diseños de Muestreo

# Muestreo aleatorio simple (MAS)
diseno_mas <- svydesign(ids = ~1, data = datos)
print(svymean(~carat, diseno_mas))

# MAS con pesos
diseno_mas_pesos <- svydesign(ids = ~1, weights = ~peso, data = datos)
print(svymean(~carat, diseno_mas_pesos))

# Muestreo estratificado
diseno_estratificado <- svydesign(ids = ~1, strata = ~estrato, weights = ~peso, data = datos)
print(svymean(~carat, diseno_estratificado))

# Muestreo por conglomerados (1 sola etapa)
diseno_conglomerados <- svydesign(ids = ~psu, data = datos)
print(svymean(~carat, diseno_conglomerados))

# Muestreo por conglomerados con pesos
diseno_conglomerados_pesos <- svydesign(ids = ~psu, weights = ~peso, data = datos)
print(svymean(~carat, diseno_conglomerados_pesos))

# Muestreo estratificado y por conglomerados (se agrega nest = TRUE)
diseno_estratificado_conglomerados <- svydesign(
  ids = ~psu,
  strata = ~estrato,
  weights = ~peso,
  data = datos,
  nest = TRUE
)
print(svymean(~carat, diseno_estratificado_conglomerados))

# Muestreo Multietápico Completo (Estratificado + UPM + USM + Pesos)
diseno_multietapico <- svydesign(
  ids = ~psu + ssu,
  strata = ~estrato,
  weights = ~peso,
  data = datos,
  nest = TRUE
)
print(svymean(~carat, diseno_multietapico))

# 3. Estimaciones Estadísticas

# Proporciones
print(svymean(~alta_gama, diseno_multietapico))

# Totales poblacionales
print(svytotal(~carat, diseno_multietapico))

# Medias por dominio (por tipo de corte)
print(svyby(~carat, ~cut, diseno_multietapico, svymean))

# 4. Modelado Avanzado con Diseño Muestral

# Regresión lineal multietápica
modelo_lineal <- svyglm(price ~ carat + depth + table, design = diseno_multietapico)
print(summary(modelo_lineal))

# Regresión logística multietápica
modelo_logistico <- svyglm(
  alta_gama ~ carat + depth,
  design = diseno_multietapico,
  family = quasibinomial()
)
print(summary(modelo_logistico))

Warning message in svydesign.default(ids = ~1, data = datos):
“No weights or probabilities supplied, assuming equal probability”


         mean     SE
carat 0.80004 0.0146
         mean     SE
carat 0.79917 0.0159
         mean     SE
carat 0.79917 0.0148


Warning message in svydesign.default(ids = ~psu, data = datos):
“No weights or probabilities supplied, assuming equal probability”


         mean     SE
carat 0.80004 0.0605
         mean     SE
carat 0.79917 0.0592
         mean     SE
carat 0.79917 0.0305
         mean     SE
carat 0.79917 0.0305
             mean     SE
alta_gama 0.49811 0.0331
       total     SE
carat 2014.9 249.03
                cut     carat         se
Fair           Fair 1.0222141 0.10466282
Good           Good 0.8804957 0.06280588
Very Good Very Good 0.7918528 0.06698714
Premium     Premium 0.9251994 0.05758060
Ideal         Ideal 0.6922130 0.06093250

Call:
svyglm(formula = price ~ carat + depth + table, design = diseno_multietapico)

Survey design:
svydesign(ids = ~psu + ssu, strata = ~estrato, weights = ~peso, 
    data = datos, nest = TRUE)

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 10131.67    4313.37   2.349   0.0261 *  
carat        7822.33     130.07  60.139   <2e-16 ***
depth        -122.38      51.56  -2.373   0.0247 *  
table         -85.62      36.88  -2.322   0.0277 *  
---
Signif. codes: 